# FNI-LLM Training Notebook v4
## Cameroon Language Model

**Run cells 1 to 9 in order. Do not skip any cell.**

- Cell 1: Mount Drive + sync code
- Cell 2: Imports + config
- Cell 3: Tokenizer
- Cell 4: Model
- Cell 5: Load data
- Cell 6: Build model
- Cell 7: Train
- Cell 8: Generate text
- Cell 9: Push to GitHub


In [ ]:
# ============================================================
# CELL 1: Mount Drive + Sync Code
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/FNI_AI_LLM
!git pull origin master

import sys
sys.path.insert(0, '/content/drive/MyDrive/FNI_AI_LLM')
print('Ready!')

In [ ]:
# ============================================================
# CELL 2: Imports + Config
# ============================================================
import re, os, json, time
import numpy as np
from collections import Counter
import torch
import torch.nn as nn

print(f'PyTorch: {torch.__version__}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

# --- CONFIG ---
LANGUAGE   = 'english'
SEQ_LEN    = 64
MAX_VOCAB  = 10000
BATCH_SIZE = 64
EPOCHS     = 50
LR         = 3e-4
WARMUP     = 5
DATA_PATH  = f'data/cameroon_languages/{LANGUAGE}/processed/{LANGUAGE}_clean.txt'

print(f'\nLanguage:  {LANGUAGE}')
print(f'Vocab:     {MAX_VOCAB:,}')
print(f'Seq len:   {SEQ_LEN}')
print(f'Epochs:    {EPOCHS}')
print(f'Batch:     {BATCH_SIZE}')

In [ ]:
# ============================================================
# CELL 3: Tokenizer
# ============================================================
class Tokenizer:
    def __init__(self, max_vocab=10000):
        self.max_vocab  = max_vocab
        self.w2i        = {}
        self.i2w        = {}
        self.vocab_size = 0

    def tokenize(self, text):
        return re.findall(r"[a-z']+|[.,!?]", text.lower())

    def build(self, corpus):
        freq  = Counter(self.tokenize(corpus))
        words = [w for w, _ in freq.most_common(self.max_vocab - 2)]
        vocab = ['<PAD>', '<UNK>'] + words
        self.w2i        = {w: i for i, w in enumerate(vocab)}
        self.i2w        = {i: w for w, i in self.w2i.items()}
        self.vocab_size = len(vocab)
        total     = sum(freq.values())
        unk_count = sum(
            c for w, c in freq.items()
            if w not in self.w2i
        )
        print(f'Vocab: {self.vocab_size:,} | '
              f'UNK: {unk_count/total:.1%} | '
              f'Tokens: {total:,}')

    def encode(self, text):
        return [self.w2i.get(w, 1)
                for w in self.tokenize(text)]

    def decode(self, ids):
        words = [self.i2w.get(i, '')
                 for i in ids
                 if i > 1 and self.i2w.get(i, '')]
        return ' '.join(words)

    def save(self, path):
        with open(path, 'w', encoding='utf-8') as f:
            json.dump({
                'w2i': self.w2i,
                'vocab_size': self.vocab_size
            }, f)

    def load(self, path):
        with open(path, encoding='utf-8') as f:
            d = json.load(f)
        self.w2i        = d['w2i']
        self.i2w        = {int(i): w for w, i in self.w2i.items()}
        self.vocab_size = d['vocab_size']

print('Tokenizer defined!')

In [ ]:
# ============================================================
# CELL 4: Transformer Model
# ============================================================
class FNITransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8,
                 d_ff=1024, num_layers=4,
                 max_seq_len=64, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding  = nn.Embedding(
            vocab_size, d_model, padding_idx=0)
        self.pos_emb    = nn.Embedding(max_seq_len, d_model)
        self.drop       = nn.Dropout(dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(
            enc_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.out  = nn.Linear(d_model, vocab_size, bias=False)
        self.out.weight = self.embedding.weight
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        T   = x.shape[1]
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        x   = self.drop(self.embedding(x) + self.pos_emb(pos))
        x   = self.transformer(x)
        return self.out(self.norm(x))

    def count_params(self):
        return sum(p.numel() for p in self.parameters())

print('FNITransformer defined!')

In [ ]:
# ============================================================
# CELL 5: Load Data
# ============================================================
with open(DATA_PATH, encoding='utf-8') as f:
    corpus = f.read()

print(f'Corpus: {len(corpus):,} chars | '
      f'{corpus.count(chr(10)):,} lines')

tok = Tokenizer(max_vocab=MAX_VOCAB)
tok.build(corpus)

os.makedirs(f'models/{LANGUAGE}', exist_ok=True)
tok.save(f'models/{LANGUAGE}/tokenizer.json')
print('Tokenizer saved!')

all_ids = tok.encode(corpus)
print(f'Encoded: {len(all_ids):,} tokens')

samples = []
for i in range(0, len(all_ids) - SEQ_LEN, SEQ_LEN // 2):
    chunk = all_ids[i: i + SEQ_LEN + 1]
    if len(chunk) == SEQ_LEN + 1:
        samples.append(chunk)

np.random.seed(42)
np.random.shuffle(samples)
n       = len(samples)
train_s = samples[:int(n * 0.8)]
val_s   = samples[int(n * 0.8): int(n * 0.9)]

print(f'Samples  — Train: {len(train_s):,} | Val: {len(val_s):,}')

In [ ]:
# ============================================================
# CELL 6: Build Model
# ============================================================
model = FNITransformer(
    vocab_size  = tok.vocab_size,
    d_model     = 256,
    num_heads   = 8,
    d_ff        = 1024,
    num_layers  = 4,
    max_seq_len = SEQ_LEN,
    dropout     = 0.1
).to(DEVICE)

print(f'Parameters: {model.count_params():,}')
print(f'Vocab size: {tok.vocab_size:,}')
print(f'Device:     {DEVICE}')
print('Model ready!')

In [ ]:
# ============================================================
# CELL 7: Training Loop
# ============================================================
def make_batches(samples, bs, shuffle=True):
    idx = np.arange(len(samples))
    if shuffle:
        np.random.shuffle(idx)
    for i in range(0, len(idx) - bs, bs):
        arr = np.array([samples[j] for j in idx[i:i+bs]])
        yield arr[:, :-1], arr[:, 1:]

def lr_schedule(epoch):
    if epoch < WARMUP:
        return LR * (epoch + 1) / WARMUP
    p = (epoch - WARMUP) / max(EPOCHS - WARMUP, 1)
    return LR * 0.5 * (1 + np.cos(np.pi * p))

opt       = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)
best_loss = float('inf')
log       = []

os.makedirs('models/checkpoints', exist_ok=True)
os.makedirs('logs', exist_ok=True)

print(f'Training {LANGUAGE} for {EPOCHS} epochs...\n')

for epoch in range(1, EPOCHS + 1):
    for g in opt.param_groups:
        g['lr'] = lr_schedule(epoch)

    # Train
    model.train()
    tl = []
    t0 = time.time()
    for X_np, y_np in make_batches(train_s, BATCH_SIZE):
        X = torch.tensor(X_np, dtype=torch.long).to(DEVICE)
        y = torch.tensor(y_np, dtype=torch.long).to(DEVICE)
        opt.zero_grad()
        loss = criterion(
            model(X).reshape(-1, tok.vocab_size),
            y.reshape(-1)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), 1.0)
        opt.step()
        tl.append(loss.item())

    # Validate
    model.eval()
    vl = []
    with torch.no_grad():
        for X_np, y_np in make_batches(
                val_s, BATCH_SIZE, shuffle=False):
            X = torch.tensor(X_np, dtype=torch.long).to(DEVICE)
            y = torch.tensor(y_np, dtype=torch.long).to(DEVICE)
            loss = criterion(
                model(X).reshape(-1, tok.vocab_size),
                y.reshape(-1)
            )
            vl.append(loss.item())

    tl_m    = float(np.mean(tl))
    vl_m    = float(np.mean(vl))
    elapsed = time.time() - t0

    if vl_m < best_loss:
        best_loss = vl_m
        torch.save({
            'model': model.state_dict(),
            'vocab': tok.w2i,
            'config': {
                'vocab_size':  tok.vocab_size,
                'd_model':     256,
                'num_heads':   8,
                'd_ff':        1024,
                'num_layers':  4,
                'max_seq_len': SEQ_LEN
            }
        }, f'models/checkpoints/{LANGUAGE}_best.pt')

    log.append({
        'epoch':      epoch,
        'train_loss': round(tl_m, 4),
        'val_loss':   round(vl_m, 4),
        'time':       round(elapsed, 2)
    })

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'train={tl_m:.4f} | '
              f'val={vl_m:.4f} | '
              f'lr={lr_schedule(epoch):.2e} | '
              f'{elapsed:.1f}s')

with open(f'logs/{LANGUAGE}_v4_training.json', 'w') as f:
    json.dump(log, f, indent=2)

print(f'\nBest val loss: {best_loss:.4f}')
print(f'Saved: models/checkpoints/{LANGUAGE}_best.pt')

In [ ]:
# ============================================================
# CELL 8: Generate Text
# ============================================================
def generate(prompt, max_new=20, temp=0.7, top_k=20):
    model.eval()
    ids  = tok.encode(prompt)
    seen = {}
    with torch.no_grad():
        for _ in range(max_new):
            x = torch.tensor(
                [ids[-SEQ_LEN:]], dtype=torch.long).to(DEVICE)
            logits = model(x)[0, -1].cpu().numpy().astype(float)

            # Block PAD and UNK
            logits[0] = -1e9
            logits[1] = -1e9

            # Repetition penalty
            for tid, cnt in seen.items():
                logits[tid] -= 3.0 * cnt

            # Top-k sampling
            logits   = logits / temp
            top_idx  = np.argsort(logits)[-top_k:]
            probs    = np.exp(
                logits[top_idx] - np.max(logits[top_idx]))
            probs   /= probs.sum()
            nxt      = int(np.random.choice(top_idx, p=probs))
            seen[nxt] = seen.get(nxt, 0) + 1
            ids.append(nxt)

    prompt_len = len(tok.encode(prompt))
    return tok.decode(ids[prompt_len:])

# Load best checkpoint
ckpt = torch.load(
    f'models/checkpoints/{LANGUAGE}_best.pt',
    map_location=DEVICE
)
model.load_state_dict(ckpt['model'])
print(f'Loaded best checkpoint\n')

prompts = [
    'cameroon is a country in central africa',
    'the people of cameroon speak',
    'douala is the largest city',
    'the official languages of cameroon are',
    'mount cameroon is the highest',
    'the history of cameroon began',
]

print(f'=== GENERATED TEXT ({LANGUAGE.upper()}) ===')
for p in prompts:
    out = generate(p, max_new=20, temp=0.7, top_k=20)
    print(f'Prompt : "{p}"')
    print(f'Output : "{out}"')
    print()

In [ ]:
# ============================================================
# CELL 9: Push to GitHub
# ============================================================
!git config --global user.email 'coursdenatationcmr@gmail.com'
!git config --global user.name 'Ronald'
%cd /content/drive/MyDrive/FNI_AI_LLM
!git add logs/ docs/visualizations/
!git commit -m '[Year3] English v4 - 10K vocab, 50 epochs, improved generation'
!git push origin master
print('Pushed to GitHub!')